In [1]:
from langchain_community.document_loaders import TextLoader

loader=TextLoader("company_policy.txt")
documents = loader.load()



C:\Users\mukes\AppData\Local\Temp\ipykernel_43784\1417238577.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [ ]:
print(documents)
print(type(documents))


[Document(metadata={'source': 'company_policy.txt'}, page_content='1. Employees must report to work on time and maintain regular attendance.\n2. All employees must follow company rules, procedures, and instructions from authorized management.\n3. Professional, respectful, and non-discriminatory behavior is expected at all times.\n4. Company property, equipment, data, and confidential information must be protected and used responsibly.\n5. Employees must not share confidential company or client information without authorization.\n6. Work responsibilities must be completed accurately, efficiently, and within agreed deadlines.\n7. Misconduct, harassment, discrimination, dishonesty, or deliberate violation of company policies will not be tolerated.\n8. Employees must maintain appropriate professional communication with colleagues, clients, and business partners.\n9. Conflicts of interest must be disclosed to management immediately.\n10. Violations of company policy may result in disciplina

In [5]:
print(documents[0].metadata)

{'source': 'company_policy.txt'}


In [6]:
documents[0].page_content

'1. Employees must report to work on time and maintain regular attendance.\n2. All employees must follow company rules, procedures, and instructions from authorized management.\n3. Professional, respectful, and non-discriminatory behavior is expected at all times.\n4. Company property, equipment, data, and confidential information must be protected and used responsibly.\n5. Employees must not share confidential company or client information without authorization.\n6. Work responsibilities must be completed accurately, efficiently, and within agreed deadlines.\n7. Misconduct, harassment, discrimination, dishonesty, or deliberate violation of company policies will not be tolerated.\n8. Employees must maintain appropriate professional communication with colleagues, clients, and business partners.\n9. Conflicts of interest must be disclosed to management immediately.\n10. Violations of company policy may result in disciplinary action, up to and including termination of employment.\nEmploye

Langchain Text Splitter for Chunking

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_spliter=RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=5
)

chunks =text_spliter.split_documents(documents)

print(len(chunks))
print(chunks)

34
[Document(metadata={'source': 'company_policy.txt'}, page_content='1. Employees must report to work on time and'), Document(metadata={'source': 'company_policy.txt'}, page_content='and maintain regular attendance.'), Document(metadata={'source': 'company_policy.txt'}, page_content='2. All employees must follow company rules,'), Document(metadata={'source': 'company_policy.txt'}, page_content='procedures, and instructions from authorized'), Document(metadata={'source': 'company_policy.txt'}, page_content='management.'), Document(metadata={'source': 'company_policy.txt'}, page_content='3. Professional, respectful, and'), Document(metadata={'source': 'company_policy.txt'}, page_content='and non-discriminatory behavior is expected at'), Document(metadata={'source': 'company_policy.txt'}, page_content='at all times.'), Document(metadata={'source': 'company_policy.txt'}, page_content='4. Company property, equipment, data, and'), Document(metadata={'source': 'company_policy.txt'}, page_con

In [19]:
print(len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- CHUNK {i} ---")
    print(chunk.page_content)

34

--- CHUNK 0 ---
1. Employees must report to work on time and

--- CHUNK 1 ---
and maintain regular attendance.

--- CHUNK 2 ---
2. All employees must follow company rules,

--- CHUNK 3 ---
procedures, and instructions from authorized

--- CHUNK 4 ---
management.

--- CHUNK 5 ---
3. Professional, respectful, and

--- CHUNK 6 ---
and non-discriminatory behavior is expected at

--- CHUNK 7 ---
at all times.

--- CHUNK 8 ---
4. Company property, equipment, data, and

--- CHUNK 9 ---
and confidential information must be protected

--- CHUNK 10 ---
and used responsibly.

--- CHUNK 11 ---
5. Employees must not share confidential company

--- CHUNK 12 ---
or client information without authorization.

--- CHUNK 13 ---
6. Work responsibilities must be completed

--- CHUNK 14 ---
accurately, efficiently, and within agreed

--- CHUNK 15 ---
deadlines.

--- CHUNK 16 ---
7. Misconduct, harassment, discrimination,

--- CHUNK 17 ---
dishonesty, or deliberate violation of company

--- CHUNK 18 ---


Embedding 
Text
↓
Embedding model
↓


For example, conceptually:

"annual leave"
        ↓
[0.12, -0.43, 0.88, 0.21, ...]

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding_model=OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector = embedding_model.embed_query(
    "How many vacation days do employees get?"
)

print(type(vector))
print(len(vector))
print(vector[:10])

<class 'list'>
1536
[-0.04541015625, 0.0226593017578125, 0.04620361328125, 0.046783447265625, -0.0028362274169921875, 0.023895263671875, -0.0029621124267578125, 0.0293121337890625, 0.0411376953125, 0.001445770263671875]


Understanding embeddings 

In [ ]:
v1 = embedding_model.embed_query(
    "Employees receive 20 days of annual leave."
)

v2 = embedding_model.embed_query(
    "Workers get twenty vacation days every year."
)

v3 = embedding_model.embed_query(
    "The company uses Python for data analysis."
)



[[0.74829908]] [[0.10764618]]


In [24]:
from sklearn.metrics.pairwise import cosine_similarity
similarity1 = cosine_similarity([v1], [v2])
similarity2=cosine_similarity([v2],[v3])


In [26]:
print(similarity2,similarity1)

[[0.10764618]] [[0.74829908]]


In [28]:
from langchain_chroma import Chroma

vector_store=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model

)

In [29]:
results = vector_store.similarity_search(
    "How many annual leave days do employees receive?",
    k=2
)

In [ ]:
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("---")

Employees receive 20 days of annual leave per
{'source': 'company_policy.txt'}
---
Employees receive a 40-minute lunch break.
{'source': 'company_policy.txt'}
---


In [32]:
# A retriever is basically a search interface over your vector store.
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}
)

question = "How many annual leave days do employees get?"

docs = retriever.invoke(question)

for doc in docs:
    print(doc.page_content)
    print("---")


Employees receive 20 days of annual leave per
---
Employees can work remotely up to three days per
---


In [36]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.

Context:
{context}

Question:
{question}
""")



In [34]:
context = "\n\n".join(doc.page_content for doc in docs)

print(context)


Employees receive 20 days of annual leave per

Employees can work remotely up to three days per


In [37]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(model="gpt-5-nano",temperature=0)

chain=rag_prompt|llm

response=chain.invoke({
    "context":context,
    "question":question
})

print(response.content)


20 days.
